In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import math

# 計算指標

In [ ]:
file_path = "btc_max_ohlc_yfinance.csv"
df = pd.read_csv(file_path)

# 2. 計算移動平均線 (20天 SMA 與 EMA)
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

# 3. 計算 RSI (14天)
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
# 使用 Wilder's 平滑法 (標準 RSI 算法)
avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()
rs = avg_gain / avg_loss
df['RSI_14'] = 100 - (100 / (1 + rs))

# 4. 計算 MACD (快線12, 慢線26, 訊號線9)
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD_Line'] = ema_12 - ema_26
df['MACD_Signal'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
df['MACD_Histogram'] = df['MACD_Line'] - df['MACD_Signal']

# 5. 計算每日報酬率
df['Daily_Return'] = df['Close'].pct_change()

# 加密貨幣交易天數設定為 365
trading_days = 365

# 6. 計算整體波動率與夏普值 (單一數值)
overall_volatility = df['Daily_Return'].std() * np.sqrt(trading_days)
mean_annual_return = df['Daily_Return'].mean() * trading_days
overall_sharpe = mean_annual_return / overall_volatility

print(f"整體年化波動率: {overall_volatility:.2%}")
print(f"整體年化夏普值: {overall_sharpe:.2f}")

# 7. 計算滾動波動率與夏普值 (如：觀察過去 365 天的動態變化)
df['Rolling_Vol_365d'] = df['Daily_Return'].rolling(window=14).std() * np.sqrt(trading_days)
df['Rolling_Return_365d'] = df['Daily_Return'].rolling(window=365).mean() * trading_days
df['Rolling_Sharpe_365d'] = df['Rolling_Return_365d'] / df['Rolling_Vol_365d']

# 儲存包含所有指標的新檔案
output_file = "btc_indicators_calculated.csv"
df.to_csv(output_file, index=False)

# 查看自相關和偏自相關

In [ ]:
# 1. 讀取與整理資料
df = pd.read_csv("btc_max_ohlc_yfinance.csv")
df['Date'] = pd.to_datetime(df['Date'], utc=True)
df = df.sort_values('Date').reset_index(drop=True)

# 2. 轉換為報酬率 (讓序列平穩化)
df['Daily_Return'] = df['Close'].pct_change()
returns = df['Daily_Return'].dropna()

# 3. 計算並取得 ACF 與 PACF 的「具體數值」 (用於機器學習特徵提取)
# nlags=40 代表計算到過去 40 天
acf_values = sm.tsa.stattools.acf(returns, nlags=40)
pacf_values = sm.tsa.stattools.pacf(returns, nlags=40, method='ywm')

# 將計算結果建立成 DataFrame 以便觀察或合併到原資料集
correlation_df = pd.DataFrame({
    'Lag': range(41),
    'ACF': acf_values,
    'PACF': pacf_values
})
print(correlation_df.head())

# 4. 繪製 ACF / PACF 圖像 (用於人工判斷 ARIMA 模型的 p, q 參數)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 繪製 ACF (判斷 MA 模型的 q 參數)
plot_acf(returns, ax=axes[0], lags=40, title='Autocorrelation (ACF) of BTC Daily Returns')
axes[0].set_xlabel('Lags')
axes[0].set_ylabel('ACF')

# 繪製 PACF (判斷 AR 模型的 p 參數)
plot_pacf(returns, ax=axes[1], lags=40, title='Partial Autocorrelation (PACF) of BTC Daily Returns', method='ywm')
axes[1].set_xlabel('Lags')
axes[1].set_ylabel('PACF')

plt.tight_layout()
plt.show()

In [ ]:
# 1. 讀取與整理資料
df = pd.read_csv("btc_max_ohlc_yfinance.csv")
df['Date'] = pd.to_datetime(df['Date'], utc=True)
df = df.sort_values('Date').reset_index(drop=True)

returns = df['Close'].dropna()

# 3. 計算並取得 ACF 與 PACF 的「具體數值」 (用於機器學習特徵提取)
# nlags=40 代表計算到過去 40 天
acf_values = sm.tsa.stattools.acf(returns, nlags=40)
pacf_values = sm.tsa.stattools.pacf(returns, nlags=40, method='ywm')

# 將計算結果建立成 DataFrame 以便觀察或合併到原資料集
correlation_df = pd.DataFrame({
    'Lag': range(41),
    'ACF': acf_values,
    'PACF': pacf_values
})
print(correlation_df.head())

# 4. 繪製 ACF / PACF 圖像 (用於人工判斷 ARIMA 模型的 p, q 參數)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 繪製 ACF (判斷 MA 模型的 q 參數)
plot_acf(returns, ax=axes[0], lags=40, title='Autocorrelation (ACF) of BTC Close price')
axes[0].set_xlabel('Lags')
axes[0].set_ylabel('ACF')

# 繪製 PACF (判斷 AR 模型的 p 參數)
plot_pacf(returns, ax=axes[1], lags=40, title='Partial Autocorrelation (PACF) of BTC Close price', method='ywm')
axes[1].set_xlabel('Lags')
axes[1].set_ylabel('PACF')

plt.tight_layout()
plt.show()

# Transformer 實驗（修正版）

這一段取代原本的兩格 Transformer 和 loss 圖，修正與補齊以下項目：

| 項目 | 做法 |
|---|---|
| 資料洩漏 | `MinMaxScaler` **只用訓練期資料擬合**，再套用到測試期 |
| 測試日期一致 | 以 8 個欄位都有值的日期為準，所有組合的訓練／測試**目標日期完全相同** |
| 切分 | 依時間 **80% 訓練 / 20% 測試（與先前實驗相同）**；loss 圖標示為 Train / **Test** loss，不稱作 Validation。需要驗證集時把 `VAL_FRAC` 設成 0.1，會從訓練期尾端切出，測試集不變 |
| 逐日預測 | 每次執行都存「日期、實際價格、預測價格、組合、seed」 |
| 基準模型 | 明日價格 = 今日價格（Naive） |
| 指標 | MSE、RMSE、MAE、MAPE；每組 5 個 seed 的平均 ± 標準差 |
| loss 紀錄 | 每個 epoch 的 train / test loss 原始數值（有驗證集時另加 val） |
| 執行環境 | GPU 型號、CUDA、主要套件版本 |
| 實驗範圍 | Close + 7 個指標（SMA、EMA、RSI、MACD 三條線、Daily Return）所有組合 = 128 組 |

**使用方式**
1. 執行階段 → 變更執行階段類型 → **T4 GPU**
2. 確認 `btc_indicators_calculated.csv` 在 Colab（上面「計算指標」那格會產生，也可以直接上傳）
3. 先用 `RUN_SCOPE = 'priority'`（4 組 × 5 seeds = 20 次）跑一輪，看結果
4. 再改成 `RUN_SCOPE = 'all'` 跑剩下的組合。結果每跑完一次就存到 Google Drive，**斷線後重新執行會自動略過已完成的部分**
   （不同切分設定存在不同子資料夾，例如 `split_80-0-20`，不會混在一起）
5. 統計和畫圖的格子隨時都能跑，會用目前已完成的結果

In [ ]:
# ==========================================
# 實驗設定（要調整只需改這一格）
# ==========================================
import os, sys, json, math, time, random, platform, subprocess, warnings
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore', message='enable_nested_tensor')

# ---- 結果存放位置：存到 Google Drive，Colab 斷線後重跑會自動略過已完成的組合 ----
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/btc_transformer_results')
else:
    BASE_DIR = Path('btc_transformer_results')

DATA_PATH = 'btc_indicators_calculated.csv'   # 由「計算指標」那格產生，或直接上傳

# ---- 特徵：Close 一定包含，另外 7 個指標任意組合 → 2^7 = 128 組 ----
TARGET_COL = 'Close'
INDICATORS = {            # 短名稱 → CSV 欄位
    'SMA':    'SMA_20',
    'EMA':    'EMA_20',
    'RSI':    'RSI_14',
    'MACD_L': 'MACD_Line',
    'MACD_S': 'MACD_Signal',
    'MACD_H': 'MACD_Histogram',
    'Ret':    'Daily_Return',
}
IND_ORDER = list(INDICATORS)
MACD3 = ['MACD_L', 'MACD_S', 'MACD_H']
PRIORITY = [                          # 報告優先的四組（MACD = 三條線一起）
    [],                               # Close
    MACD3 + ['Ret'],                  # Close + MACD + Return
    ['RSI'] + MACD3,                  # Close + MACD + RSI
    ['RSI'] + MACD3 + ['Ret'],        # Close + MACD + RSI + Return
]

RUN_SCOPE = 'priority'     # 先用 'priority' 跑 4 組 × 5 seeds；確認沒問題再改 'all' 跑完 128 組
SEEDS = [42, 43, 44, 45, 46]

# ---- 時間序列切分（依「預測目標日期」先後）----
LOOKBACK = 60
TEST_FRAC = 0.20   # 測試集 = 最後 20%（與先前實驗的 8/2 相同）
VAL_FRAC = 0.0     # 0   → 訓練 80% / 測試 20%，與先前實驗相同（預設）
                   # 0.1 → 從訓練的 80% 尾端再切 10% 當驗證集（70/10/20），測試集不變

# ---- 模型與訓練（架構、超參數與原本相同）----
NUM_LAYERS, NHEAD, HIDDEN_DIM, DROPOUT = 2, 4, 128, 0.1
LR = 0.001
NUM_EPOCHS = 300
SELECT_BEST_VAL = True     # 只在 VAL_FRAC > 0 時有作用：採用驗證 loss 最低那個 epoch 的權重
                           # VAL_FRAC = 0 時與原本相同，用第 NUM_EPOCHS 個 epoch 的權重

# 每種切分設定存在自己的子資料夾，避免不同設定的結果混在一起續跑
SPLIT_TAG = f'split_{round(100 * (1 - TEST_FRAC - VAL_FRAC))}-{round(100 * VAL_FRAC)}-{round(100 * TEST_FRAC)}'
OUT_DIR = BASE_DIR / SPLIT_TAG
PRED_DIR, LOSS_DIR, FIG_DIR = OUT_DIR / 'predictions', OUT_DIR / 'loss', OUT_DIR / 'figures'
for d in (PRED_DIR, LOSS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用設備：', device, '｜結果資料夾：', OUT_DIR)
if device.type == 'cpu':
    print('⚠️ 目前沒有 GPU：請到「執行階段 → 變更執行階段類型」選 T4 GPU')

# ---- 組合命名（MACD 三條線都在時合併顯示為 MACD）----
def combo_name(inds):
    s = set(inds)
    full_macd = set(MACD3) <= s
    parts = ['Close']
    for k in IND_ORDER:
        if full_macd and k in MACD3:
            if k == 'MACD_L':
                parts.append('MACD')
        elif k in s:
            parts.append(k)
    return '+'.join(parts)

def combo_key(inds):            # 檔名用
    return combo_name(inds).replace('+', '-')

def combo_columns(inds):
    s = set(inds)
    return [TARGET_COL] + [INDICATORS[k] for k in IND_ORDER if k in s]

PRIORITY_NAMES = [combo_name(c) for c in PRIORITY]
ALL_COMBOS = [list(c) for r in range(len(IND_ORDER) + 1) for c in combinations(IND_ORDER, r)]
ALL_COMBOS = PRIORITY + [c for c in ALL_COMBOS if combo_name(c) not in PRIORITY_NAMES]   # 優先組合先跑
print('優先組合：', PRIORITY_NAMES)
print('全部組合數：', len(ALL_COMBOS))

## 執行環境紀錄
每次開新的 Colab session 都會在 `environment_log.jsonl` 追加一筆；每次實驗的 GPU 型號也會寫進 `runs_metrics.csv`（跨多個 session 跑時可能分到不同 GPU）。

In [ ]:
import sklearn, matplotlib, statsmodels

def collect_environment():
    cuda = torch.cuda.is_available()
    env = {
        'recorded_at':      pd.Timestamp.now(tz='Asia/Taipei').isoformat(timespec='seconds'),
        'python':           sys.version.split()[0],
        'platform':         platform.platform(),
        'torch':            torch.__version__,
        'torch_cuda_build': torch.version.cuda,
        'cudnn':            torch.backends.cudnn.version() if cuda else None,
        'numpy':            np.__version__,
        'pandas':           pd.__version__,
        'scikit-learn':     sklearn.__version__,
        'matplotlib':       matplotlib.__version__,
        'statsmodels':      statsmodels.__version__,
        'device':           str(device),
        'gpu':              torch.cuda.get_device_name(0) if cuda else 'CPU only',
    }
    try:
        env['nvidia_smi'] = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=10).stdout.strip() or None
    except Exception:
        env['nvidia_smi'] = None
    return env

ENV = collect_environment()
with open(OUT_DIR / 'environment_log.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(ENV, ensure_ascii=False) + '\n')
for k, v in ENV.items():
    print(f'{k:17s}: {v}')

## 資料準備
- 只保留 8 個欄位都有值的日期（SMA_20 前 19 天是空值），**所有組合共用同一批日期**
- 依預測目標日期切 **80% 訓練 / 20% 測試**（`VAL_FRAC > 0` 時從訓練期尾端再切出驗證集）
- **scaler 只用訓練期資料擬合**：`scaler_X` 用訓練期所有列，`scaler_y` 用訓練期的目標收盤價
- 基準模型：明日價格 = 今日價格

In [ ]:
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'].astype(str).str[:10])
df = df.sort_values('Date').reset_index(drop=True)

ALL_COLS = [TARGET_COL] + list(INDICATORS.values())
df = df.dropna(subset=ALL_COLS).reset_index(drop=True)

n_samples = len(df) - LOOKBACK
n_trainval = int(n_samples * (1 - TEST_FRAC))      # 與原本 train_size = int(len(X) * 0.8) 相同
n_val = int(n_samples * VAL_FRAC)
n_train = n_trainval - n_val
n_test = n_samples - n_trainval
USE_VAL = n_val > 0
SPLITS = ['train', 'val', 'test'] if USE_VAL else ['train', 'test']
target_rows = np.arange(LOOKBACK, len(df))                 # 第 k 個樣本要預測的那一天（列號）
split = np.array(['train'] * n_train + ['val'] * n_val + ['test'] * n_test)
train_last_row = target_rows[n_train - 1]

# ---- 修正資料洩漏：scaler 只看訓練期 ----
scaler_X = MinMaxScaler(feature_range=(0, 1)).fit(df.loc[:train_last_row, ALL_COLS].values)
scaler_y = MinMaxScaler(feature_range=(0, 1)).fit(df.loc[target_rows[:n_train], [TARGET_COL]].values)
X_scaled = scaler_X.transform(df[ALL_COLS].values).astype(np.float32)
y_scaled = scaler_y.transform(df[[TARGET_COL]].values).astype(np.float32)[:, 0]

# 滑動視窗：用前 LOOKBACK 天（不含當天）預測當天收盤價，與原本 create_sequences 相同
windows = np.stack([X_scaled[r - LOOKBACK:r] for r in target_rows])     # (樣本數, LOOKBACK, 8)
y_target = y_scaled[target_rows]

close = df[TARGET_COL].values
samples = pd.DataFrame({
    'target_date': df['Date'].values[target_rows],
    'split':       split,
    'actual':      close[target_rows],
    'naive':       close[target_rows - 1],                  # 基準：明日價格 = 今日價格
})

def calc_metrics(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    err = pred - actual
    return {'MSE':  float(np.mean(err ** 2)),
            'RMSE': float(np.sqrt(np.mean(err ** 2))),
            'MAE':  float(np.mean(np.abs(err))),
            'MAPE': float(np.mean(np.abs(err) / np.abs(actual)) * 100)}

naive_metrics = {sp: calc_metrics(samples.loc[samples.split == sp, 'actual'],
                                  samples.loc[samples.split == sp, 'naive']) for sp in SPLITS}

(samples[['target_date', 'split', 'actual', 'naive']]
    .rename(columns={'naive': 'predicted'})
    .assign(model='Naive (tomorrow = today)')
    .to_csv(OUT_DIR / 'naive_baseline_predictions.csv', index=False))

# ---- 切分資訊 ----
split_info = {}
for sp in SPLITS:
    g = samples[samples.split == sp]
    split_info[sp] = {'n': int(len(g)),
                      'first_target_date': f'{g.target_date.min():%Y-%m-%d}',
                      'last_target_date':  f'{g.target_date.max():%Y-%m-%d}',
                      'close_min': float(g.actual.min()), 'close_max': float(g.actual.max())}
    print(f"{sp:5s} {len(g):5d} 天  {split_info[sp]['first_target_date']} ~ {split_info[sp]['last_target_date']}"
          f"   Close {g.actual.min():>10,.0f} ~ {g.actual.max():>10,.0f}")
train_max = float(scaler_y.data_max_[0])
pct_above = float((samples.loc[samples.split == 'test', 'actual'] > train_max).mean() * 100)
split_info.update({'lookback': LOOKBACK, 'train_close_max_scaler_upper_bound': train_max,
                   'test_days_above_train_max_pct': pct_above})
with open(OUT_DIR / 'split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)

print(f'\n訓練期最高價（scaler 上限）：{train_max:,.0f}')
print(f'測試期有 {pct_above:.1f}% 的天數高於這個價格 → 縮放後 > 1，模型必須外插')
print('\nNaive 基準（明日 = 今日）：')
display(pd.DataFrame(naive_metrics).T.style.format({'MSE': '{:.3e}', 'RMSE': '{:,.0f}', 'MAE': '{:,.0f}', 'MAPE': '{:.2f}'}))

In [ ]:
# ==========================================
# 儲存每次實驗結果（支援中斷續跑）
# ==========================================
RUNS_CSV = OUT_DIR / 'runs_metrics.csv'

def completed_runs():
    if not RUNS_CSV.exists():
        return set()
    r = pd.read_csv(RUNS_CSV)
    return set(zip(r['combo'], r['seed']))

def record_run(inds, seed, pred_usd, loss_hist, best_epoch, train_time):
    """存逐日預測、每個 epoch 的 loss，並把指標追加到 runs_metrics.csv（最後才寫，確保中斷時會重跑）"""
    name, key = combo_name(inds), combo_key(inds)
    pred_df = samples[['target_date', 'split', 'actual']].copy()
    pred_df['predicted'] = pred_usd
    pred_df['combo'] = name
    pred_df['seed'] = seed
    pred_df.to_csv(PRED_DIR / f'{key}__seed{seed}.csv', index=False)

    pd.DataFrame(loss_hist, columns=['epoch', 'train_loss', 'val_loss', 'test_loss']) \
      .to_csv(LOSS_DIR / f'{key}__seed{seed}.csv', index=False)

    row = {'combo': name, 'combo_key': key, 'n_indicators': len(inds),
           'features': '|'.join(combo_columns(inds)), 'seed': seed,
           'epochs': len(loss_hist), 'best_epoch': best_epoch,
           'selected_weights': 'best_val_epoch' if (USE_VAL and SELECT_BEST_VAL) else 'last_epoch',
           'train_time_s': round(train_time, 1), 'gpu': ENV['gpu']}
    for sp in SPLITS:
        m = pred_df['split'] == sp
        for k, v in calc_metrics(pred_df.loc[m, 'actual'], pred_df.loc[m, 'predicted']).items():
            row[f'{sp}_{k}'] = v
    pd.DataFrame([row]).to_csv(RUNS_CSV, mode='a', header=not RUNS_CSV.exists(), index=False)
    return row

In [ ]:
# ==========================================
# 圖表共用設定（報告用英文標籤，避免 Colab 中文字型變方塊）
# ==========================================
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e4e3df', 'grid.linewidth': 0.6,
    'axes.edgecolor': '#8a8984', 'axes.labelcolor': '#2b2b2a', 'axes.titlesize': 10.5,
    'xtick.color': '#52514e', 'ytick.color': '#52514e',
    'legend.frameon': False, 'legend.fontsize': 8.5, 'lines.linewidth': 1.5,
})
COL_ACTUAL = '#1f1f1e'      # 實際價格
COL_NAIVE = '#8a8984'       # 基準 / 參考線
COL_TRAIN = '#52514e'       # 訓練 loss
COL_OTHER = '#a9a8a2'       # 非優先組合
COMBO_COLORS = dict(zip(PRIORITY_NAMES, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']))

SPLIT_BOUNDS = {sp: (samples.loc[samples.split == sp, 'target_date'].min(),
                     samples.loc[samples.split == sp, 'target_date'].max()) for sp in SPLITS}
usd_k = FuncFormatter(lambda v, _: f'${v/1000:,.0f}k' if v >= 1000 else f'${v:,.0f}')

def shade_splits(ax, label=True):
    for sp, c, txt in [('val', '#f1f0ec', 'Validation'), ('test', '#e4e3de', 'Test')]:
        if sp not in SPLITS:
            continue
        a, b = SPLIT_BOUNDS[sp]
        ax.axvspan(a, b, color=c, zorder=0, lw=0)
        if label:
            ax.text(a + (b - a) / 2, 1.01, txt, transform=ax.get_xaxis_transform(),
                    ha='center', va='bottom', fontsize=8.5, color='#52514e')

def date_axis(ax):
    loc = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))

def save(fig, name):
    fig.savefig(FIG_DIR / name, bbox_inches='tight')
    plt.show()

# ---- 圖 1：資料切分 ----
fig, ax = plt.subplots(figsize=(11, 4))
shade_splits(ax)
ax.plot(df['Date'], df['Close'], color=COL_ACTUAL, lw=1.1, label='BTC close')
ax.axhline(train_max, color=COL_NAIVE, ls='--', lw=1,
           label=f'Training-period max = scaler upper bound (${train_max:,.0f})')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(usd_k)
ax.set_ylabel('Close (USD, log scale)')
date_axis(ax)
split_txt = ' / '.join(f'{lbl} {n}' for lbl, n in [('train', n_train), ('validation', n_val), ('test', n_test)] if n > 0)
ax.set_title(f'Chronological split by target date — {split_txt} days', loc='left', pad=16)
ax.legend(loc='lower right')
save(fig, '01_data_split.png')

## 模型與單次實驗
模型架構與原本完全相同。每個 epoch 記錄：
- `train_loss`：訓練模式（有 dropout）下的訓練集 MSE，也就是實際被最佳化的 loss
- `test_loss`：測試集 MSE，**只記錄、不參與訓練或任何選擇**；圖上標示為 Test loss，不能稱作 Validation loss
- `val_loss`：只有 `VAL_FRAC > 0` 時才有，用來挑選最佳 epoch；`VAL_FRAC = 0` 時為空值，與原本一樣用第 300 個 epoch 的權重

loss 都是在縮放後（0–1）的尺度上計算的。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0)]
        return x

class TransformerTimeSeries(nn.Module):
    def __init__(self, feature_size=1, num_layers=2, nhead=4, hidden_dim=128, dropout=0.1):
        super(TransformerTimeSeries, self).__init__()
        self.model_type = 'Transformer'
        self.input_linear = nn.Linear(feature_size, hidden_dim)
        self.pos_encoder = PositionalEncoding(d_model=hidden_dim)
        encoder_layers = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.decoder = nn.Linear(hidden_dim, 1)
        self.hidden_dim = hidden_dim

    def forward(self, src):
        src = self.input_linear(src)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src)
        # 取最後一個時間步的輸出
        output = self.decoder(output[-1, :, :])
        return output

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 所有視窗一次搬上 GPU，形狀 (seq_len, 樣本數, 8)，各組合再挑欄位
X_all_t = torch.from_numpy(windows).permute(1, 0, 2).contiguous().to(device)
y_all_t = torch.from_numpy(y_target).to(device)
SL = {'train': slice(0, n_train), 'val': slice(n_train, n_train + n_val), 'test': slice(n_train + n_val, n_samples)}

def run_experiment(inds, seed):
    set_seed(seed)
    col_idx = [0] + [1 + IND_ORDER.index(k) for k in IND_ORDER if k in set(inds)]
    X = X_all_t.index_select(2, torch.tensor(col_idx, device=device))
    Xtr, ytr = X[:, SL['train']], y_all_t[SL['train']]
    Xte, yte = X[:, SL['test']], y_all_t[SL['test']]
    if USE_VAL:
        Xva, yva = X[:, SL['val']], y_all_t[SL['val']]

    model = TransformerTimeSeries(feature_size=len(col_idx), num_layers=NUM_LAYERS, nhead=NHEAD,
                                  hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    hist, best_val, best_epoch, best_state = [], float('inf'), None, None
    t0 = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(Xtr).squeeze(-1), ytr)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(Xva).squeeze(-1), yva).item() if USE_VAL else float('nan')
            test_loss = criterion(model(Xte).squeeze(-1), yte).item()   # 只記錄
        hist.append((epoch, loss.item(), val_loss, test_loss))
        if USE_VAL and val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    if USE_VAL and SELECT_BEST_VAL:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred_scaled = model(X).squeeze(-1).cpu().numpy()
    pred_usd = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1))[:, 0]
    return pred_usd, hist, best_epoch, time.time() - t0

## 批次執行（可中斷續跑）
已完成的 (組合, seed) 會從 Drive 上的 `runs_metrics.csv` 讀取並略過。Colab 斷線後重新連線，從頭依序執行到這格即可接著跑。

In [ ]:
done = completed_runs()
combos = PRIORITY if RUN_SCOPE == 'priority' else ALL_COMBOS
todo = [(c, s) for c in combos for s in SEEDS if (combo_name(c), s) not in done]
total = len(combos) * len(SEEDS)
print(f'範圍：{RUN_SCOPE}｜共 {total} 次，已完成 {total - len(todo)}，待跑 {len(todo)}')

t_start = time.time()
for i, (inds, seed) in enumerate(todo, 1):
    pred_usd, hist, best_epoch, sec = run_experiment(inds, seed)
    row = record_run(inds, seed, pred_usd, hist, best_epoch, sec)
    eta_min = (time.time() - t_start) / i * (len(todo) - i) / 60
    be_txt = f'best_val_epoch={best_epoch:3d}  ' if best_epoch else ''
    print(f"[{i:3d}/{len(todo)}] {row['combo']:<36s} seed={seed}  {be_txt}"
          f"test MAE={row['test_MAE']:>8,.0f}  MAPE={row['test_MAPE']:6.2f}%  ({sec:.0f}s，剩餘約 {eta_min:.0f} 分)")
print('完成')

## 彙整結果
- `metrics_summary.csv`：每個組合 5 個 seed 的測試集（及驗證集，若有）平均與標準差（樣本標準差，ddof=1），依測試 MAPE 排序，並標記是否勝過 Naive
- `report_table_priority.csv`：四個優先組合 + Naive 的報告用表格

In [ ]:
runs = pd.read_csv(RUNS_CSV).drop_duplicates(['combo', 'seed'], keep='last')
METRICS = ['MSE', 'RMSE', 'MAE', 'MAPE']

agg = {}
for sp in [sp for sp in SPLITS if sp != 'train']:
    for m in METRICS:
        agg[f'{sp}_{m}_mean'] = (f'{sp}_{m}', 'mean')
        agg[f'{sp}_{m}_std'] = (f'{sp}_{m}', 'std')
summary = (runs.groupby(['combo', 'combo_key', 'n_indicators', 'features'])
               .agg(n_seeds=('seed', 'nunique'), best_epoch_median=('best_epoch', 'median'), **agg)
               .reset_index()
               .sort_values('test_MAPE_mean').reset_index(drop=True))
summary.insert(0, 'rank_by_test_MAPE', np.arange(1, len(summary) + 1))
for m in METRICS:
    summary[f'beats_naive_{m}'] = summary[f'test_{m}_mean'] < naive_metrics['test'][m]
summary.to_csv(OUT_DIR / 'metrics_summary.csv', index=False)

def pm(mean, std, fmt):
    return f'{mean:{fmt}} ± {std:{fmt}}' if pd.notna(std) else f'{mean:{fmt}}'

rows = []
for name in PRIORITY_NAMES:
    r = summary[summary.combo == name]
    if r.empty:
        continue
    r = r.iloc[0]
    rows.append({'Model': f'Transformer: {name}', 'Seeds': int(r.n_seeds),
                 'Test MSE': pm(r.test_MSE_mean, r.test_MSE_std, '.3e'),
                 'Test RMSE (USD)': pm(r.test_RMSE_mean, r.test_RMSE_std, ',.0f'),
                 'Test MAE (USD)': pm(r.test_MAE_mean, r.test_MAE_std, ',.0f'),
                 'Test MAPE (%)': pm(r.test_MAPE_mean, r.test_MAPE_std, '.2f')})
    if USE_VAL:
        rows[-1]['Best val epoch (median)'] = int(r.best_epoch_median)
nm = naive_metrics['test']
rows.append({'Model': 'Naive (tomorrow = today)', 'Seeds': '-',
             'Test MSE': f"{nm['MSE']:.3e}", 'Test RMSE (USD)': f"{nm['RMSE']:,.0f}",
             'Test MAE (USD)': f"{nm['MAE']:,.0f}", 'Test MAPE (%)': f"{nm['MAPE']:.2f}"})
report_table = pd.DataFrame(rows)
report_table.to_csv(OUT_DIR / 'report_table_priority.csv', index=False)
print(f'已完成 {len(runs)} 次實驗、{len(summary)} 個組合')
display(report_table)

if len(summary) > len(PRIORITY_NAMES):
    print('\n全部組合前 10 名（依測試 MAPE）：')
    display(summary[['rank_by_test_MAPE', 'combo', 'n_seeds', 'test_MAE_mean', 'test_MAE_std',
                     'test_MAPE_mean', 'test_MAPE_std', 'beats_naive_MAPE']].head(10)
            .style.format({'test_MAE_mean': '{:,.0f}', 'test_MAE_std': '{:,.0f}',
                           'test_MAPE_mean': '{:.2f}', 'test_MAPE_std': '{:.2f}'}))

## 匯出逐日預測
- `test_predictions_wide.csv`：測試期每一天的實際價格、Naive、四個優先組合的預測（seed 平均與標準差）
- `predictions_priority_long.csv`：四個優先組合 × 每個 seed × 全部日期（含訓練／驗證期）
- `predictions_test_all_runs.csv`：所有已完成實驗的測試期逐日預測
- 個別實驗的原始檔在 `predictions/`、`loss/` 資料夾

In [ ]:
def load_runs_files(folder, name):
    key = name.replace('+', '-')
    seeds = sorted(runs.loc[runs.combo == name, 'seed'])
    if not seeds:
        return None
    parts = []
    for s in seeds:
        d = pd.read_csv(folder / f'{key}__seed{s}.csv')
        d['seed'] = s
        parts.append(d)
    out = pd.concat(parts, ignore_index=True)
    if 'target_date' in out:
        out['target_date'] = pd.to_datetime(out['target_date'])
    return out

preds = {n: load_runs_files(PRED_DIR, n) for n in PRIORITY_NAMES}
losses = {n: load_runs_files(LOSS_DIR, n) for n in PRIORITY_NAMES}

def seed_stats(p, sp=None):
    """回傳以 samples 日期對齊的 seed 平均與標準差"""
    sub = p if sp is None else p[p.split == sp]
    piv = sub.pivot(index='target_date', columns='seed', values='predicted')
    dates = samples.target_date if sp is None else samples.loc[samples.split == sp, 'target_date']
    piv = piv.reindex(dates)
    return piv.mean(axis=1).values, piv.std(axis=1).values

full = samples[['target_date', 'split', 'actual', 'naive']].reset_index(drop=True)
for n, p in preds.items():
    if p is not None:
        full[f'{n} mean'], full[f'{n} std'] = seed_stats(p)
test = full[full.split == 'test'].drop(columns='split').reset_index(drop=True)
test.to_csv(OUT_DIR / 'test_predictions_wide.csv', index=False)

pd.concat([p for p in preds.values() if p is not None]) \
  .to_csv(OUT_DIR / 'predictions_priority_long.csv', index=False)

EXPORT_ALL_TEST = True   # 128 組都跑完時要讀 640 個檔，約需 1–2 分鐘
if EXPORT_ALL_TEST:
    parts = []
    for _, r in runs.iterrows():
        d = pd.read_csv(PRED_DIR / f"{r.combo_key}__seed{r.seed}.csv")
        parts.append(d[d.split == 'test'].drop(columns='split'))
    pd.concat(parts, ignore_index=True).to_csv(OUT_DIR / 'predictions_test_all_runs.csv', index=False)
print('已匯出')
test.head()

## 報告圖表
全部存在 `figures/`（200 dpi PNG）。

In [ ]:
# ---- 圖 2：loss 曲線（對數尺度）----
# 沒有驗證集時畫 Train vs Test，標示為 Test（測試 loss 只記錄，不參與訓練或選擇）
SECOND_LOSS = ('val_loss', 'Validation') if USE_VAL else ('test_loss', 'Test')
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, name in zip(axes.flat, PRIORITY_NAMES):
    L = losses[name]
    if L is None:
        ax.set_title(f'{name} (not run yet)', loc='left'); continue
    for col, color, label in [('train_loss', COL_TRAIN, 'Train'), (SECOND_LOSS[0], COMBO_COLORS[name], SECOND_LOSS[1])]:
        piv = L.pivot(index='epoch', columns='seed', values=col)
        ax.fill_between(piv.index, piv.min(axis=1), piv.max(axis=1), color=color, alpha=0.18, lw=0)
        ax.plot(piv.index, piv.mean(axis=1), color=color, label=f'{label} (mean of seeds)')
    if USE_VAL:
        be = runs.loc[runs.combo == name, 'best_epoch'].median()
        ax.axvline(be, color=COL_NAIVE, ls=':', lw=1)
        ax.text(be, 0.97, f' best val epoch (median) = {be:.0f}', transform=ax.get_xaxis_transform(),
                fontsize=8, color='#52514e', va='top')
    ax.set_yscale('log')
    ax.set_title(f'{name}  (n = {L.seed.nunique()} seeds)', loc='left')
    ax.legend(loc='lower left')
for ax in axes[1]: ax.set_xlabel('Epoch')
for ax in axes[:, 0]: ax.set_ylabel('MSE loss (scaled, log scale)')
if USE_VAL:
    fig.suptitle('Training vs validation loss — line = mean, band = min–max across seeds', x=0.01, ha='left')
else:
    fig.suptitle('Training vs test loss — line = mean, band = min–max across seeds\n'
                 'Test loss is recorded for reference only; it is not used for training or model selection.',
                 x=0.01, ha='left')
fig.tight_layout()
save(fig, '02_loss_curves_log.png')

In [ ]:
# ---- 圖 3：測試期放大圖（實際 vs 預測，seed 平均 ± 1 std）----
def mape_note(name):
    r = summary[summary.combo == name]
    if r.empty: return ''
    r = r.iloc[0]
    return (f"Model MAPE {pm(r.test_MAPE_mean, r.test_MAPE_std, '.2f')}%\n"
            f"Naive MAPE {naive_metrics['test']['MAPE']:.2f}%")

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, name in zip(axes.flat, PRIORITY_NAMES):
    ax.plot(test.target_date, test.actual, color=COL_ACTUAL, lw=1.1, label='Actual')
    if f'{name} mean' in test:
        c = COMBO_COLORS[name]
        m, s = test[f'{name} mean'], test[f'{name} std'].fillna(0)
        ax.fill_between(test.target_date, m - s, m + s, color=c, alpha=0.22, lw=0, label='±1 std across seeds')
        ax.plot(test.target_date, m, color=c, lw=1.4, label='Predicted (mean of seeds)')
        ax.text(0.99, 0.03, mape_note(name), transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5, color='#2b2b2a')
    else:
        name += ' (not run yet)'
    ax.set_title(name, loc='left')
    ax.yaxis.set_major_formatter(usd_k)
    date_axis(ax)
axes[0, 0].legend(loc='upper left')
for ax in axes[:, 0]: ax.set_ylabel('BTC close (USD)')
fig.suptitle(f"Test period {split_info['test']['first_target_date']} – {split_info['test']['last_target_date']}: actual vs predicted",
             x=0.01, ha='left')
fig.tight_layout()
save(fig, '03_test_period_zoom.png')

# ---- 圖 4：測試期最後 90 天（看得出預測是否只是落後一天）----
last = test.tail(90)
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, name in zip(axes.flat, PRIORITY_NAMES):
    ax.plot(last.target_date, last.actual, color=COL_ACTUAL, lw=1.2, label='Actual')
    ax.plot(last.target_date, last.naive, color=COL_NAIVE, lw=1.1, ls='--', label='Naive (tomorrow = today)')
    if f'{name} mean' in last:
        ax.plot(last.target_date, last[f'{name} mean'], color=COMBO_COLORS[name], lw=1.5, marker='o', ms=2.5,
                label='Predicted (mean of seeds)')
    ax.set_title(name, loc='left')
    ax.yaxis.set_major_formatter(usd_k)
    date_axis(ax)
axes[0, 0].legend(loc='best')
for ax in axes[:, 0]: ax.set_ylabel('BTC close (USD)')
fig.suptitle('Last 90 days of the test period', x=0.01, ha='left')
fig.tight_layout()
save(fig, '04_test_last90_zoom.png')

In [ ]:
# ---- 圖 5：四個優先組合的測試指標（平均 ± 1 std）vs Naive ----
avail = [n for n in PRIORITY_NAMES if n in set(summary.combo)]
sub = summary.set_index('combo').loc[avail]
x = np.arange(len(avail))
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, m in zip(axes, ['RMSE', 'MAE', 'MAPE']):
    ax.bar(x, sub[f'test_{m}_mean'], yerr=sub[f'test_{m}_std'].fillna(0), width=0.6, capsize=4,
           color=[COMBO_COLORS[n] for n in avail], error_kw=dict(lw=1, ecolor='#2b2b2a'))
    ax.axhline(naive_metrics['test'][m], color=COL_NAIVE, ls='--', lw=1.2, label='Naive (tomorrow = today)')
    ax.set_xticks(x, [n.replace('+', '+\n') for n in avail], fontsize=8)
    ax.set_title(f"Test {m} ({'%' if m == 'MAPE' else 'USD'})", loc='left')
    ax.grid(axis='x', visible=False)
    if m != 'MAPE':
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper right', bbox_to_anchor=(0.99, 1.0))
fig.suptitle('Test-set error by feature set — bar = mean, whisker = ±1 std across seeds', x=0.01, ha='left')
fig.tight_layout()
save(fig, '05_metrics_priority.png')

In [ ]:
# ---- 圖 6–7：全部組合（跑完 RUN_SCOPE='all' 後才有意義）----
if len(summary) <= len(PRIORITY_NAMES):
    print("目前只有優先組合的結果；把 RUN_SCOPE 改成 'all' 跑完後再執行這格。")
else:
    # 圖 6：全部組合依測試 MAPE 排名
    fig, ax = plt.subplots(figsize=(12, 4.5))
    other = summary[~summary.combo.isin(PRIORITY_NAMES)]
    ax.errorbar(other.rank_by_test_MAPE, other.test_MAPE_mean, yerr=other.test_MAPE_std.fillna(0),
                fmt='o', ms=3.5, color=COL_OTHER, ecolor='#d6d5cf', elinewidth=0.8, label='Other combinations')
    for name in PRIORITY_NAMES:
        r = summary[summary.combo == name]
        if r.empty: continue
        ax.errorbar(r.rank_by_test_MAPE, r.test_MAPE_mean, yerr=r.test_MAPE_std.fillna(0), fmt='o', ms=8,
                    color=COMBO_COLORS[name], mec='white', mew=1.5, ecolor=COMBO_COLORS[name], elinewidth=1.2,
                    label=f"{name} (rank {int(r.rank_by_test_MAPE.iloc[0])})", zorder=5)
    ax.axhline(naive_metrics['test']['MAPE'], color=COL_NAIVE, ls='--', lw=1.2, label='Naive (tomorrow = today)')
    ax.set_xlabel('Rank (sorted by mean test MAPE)')
    ax.set_ylabel('Test MAPE (%)')
    ax.set_title(f'All {len(summary)} feature sets (Close + any subset of 7 indicators) — dot = mean, whisker = ±1 std',
                 loc='left')
    ax.legend(loc='upper left', ncol=2)
    save(fig, '06_all_combos_rank.png')

    # 圖 7：每個指標的邊際效果 + 指標數量
    mape = dict(zip(summary.combo, summary.test_MAPE_mean))
    rows = []
    for inds in ALL_COMBOS:
        for k in IND_ORDER:
            if k in inds: continue
            a, b = combo_name(inds), combo_name(inds + [k])
            if a in mape and b in mape:
                rows.append({'indicator': k, 'delta': mape[b] - mape[a]})
    eff = pd.DataFrame(rows).groupby('indicator')['delta'].agg(['mean', 'std', 'count']).reindex(IND_ORDER).dropna(subset=['mean'])
    eff.to_csv(OUT_DIR / 'indicator_marginal_effect.csv')

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={'width_ratios': [1, 1.2]})
    yy = np.arange(len(eff))
    a1.barh(yy, eff['mean'], xerr=eff['std'], color='#2a78d6', height=0.55, capsize=3, error_kw=dict(lw=1, ecolor='#2b2b2a'))
    a1.axvline(0, color=COL_ACTUAL, lw=0.8)
    a1.set_yticks(yy, eff.index); a1.invert_yaxis()
    a1.set_xlabel('Change in test MAPE when the indicator is added (pp)\n← improves     worsens →')
    a1.set_title('Marginal effect of each indicator (paired comparison, mean ± std)', loc='left')
    a1.grid(axis='y', visible=False)

    groups = [summary.loc[summary.n_indicators == k, 'test_MAPE_mean'].values for k in range(len(IND_ORDER) + 1)]
    a2.boxplot(groups, positions=range(len(groups)), widths=0.5, showfliers=False,
               medianprops=dict(color='#2a78d6', lw=1.5), boxprops=dict(color='#52514e'),
               whiskerprops=dict(color='#52514e'), capprops=dict(color='#52514e'))
    rng = np.random.default_rng(0)
    for k, g in enumerate(groups):
        a2.scatter(k + rng.uniform(-0.12, 0.12, len(g)), g, s=10, color=COL_OTHER, zorder=3)
    a2.axhline(naive_metrics['test']['MAPE'], color=COL_NAIVE, ls='--', lw=1.2, label='Naive')
    a2.set_xlabel('Number of indicators added to Close')
    a2.set_ylabel('Test MAPE (%), mean of seeds')
    a2.set_title('Test MAPE by number of indicators', loc='left')
    a2.legend(loc='upper left')
    fig.tight_layout()
    save(fig, '07_indicator_effect.png')

In [ ]:
# ---- 圖 8：全期間（訓練 / 驗證 / 測試）實際 vs 預測 ----
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, name in zip(axes.flat, PRIORITY_NAMES):
    shade_splits(ax, label=(ax in axes[0]))
    ax.plot(full.target_date, full.actual, color=COL_ACTUAL, lw=1.0, label='Actual')
    if f'{name} mean' in full:
        ax.plot(full.target_date, full[f'{name} mean'], color=COMBO_COLORS[name], lw=1.1, alpha=0.9,
                label='Predicted (mean of seeds)')
    ax.axhline(train_max, color=COL_NAIVE, ls='--', lw=0.9, label='Training-period max')
    ax.set_yscale('log')
    ax.yaxis.set_major_formatter(usd_k)
    ax.set_title(name, loc='left', pad=14 if ax in axes[0] else 6)
    date_axis(ax)
axes[1, 0].legend(loc='lower right')
for ax in axes[:, 0]: ax.set_ylabel('BTC close (USD, log scale)')
fig.tight_layout()
save(fig, '08_full_period.png')

## 產出檔案一覽（Google Drive：`btc_transformer_results/split_80-0-20/`）

| 檔案 | 內容 |
|---|---|
| `environment_log.jsonl` | 每個 session 的 GPU、CUDA、套件版本 |
| `split_info.json` | 各切分的日期範圍、天數、價格範圍、scaler 上限 |
| `runs_metrics.csv` | 每次實驗（組合 × seed）的 train/val/test MSE、RMSE、MAE、MAPE、best epoch、訓練時間、GPU |
| `metrics_summary.csv` | 每個組合的平均 ± 標準差、排名、是否勝過 Naive |
| `report_table_priority.csv` | 四個優先組合 + Naive 報告表 |
| `naive_baseline_predictions.csv` | 基準模型逐日預測 |
| `test_predictions_wide.csv` | 測試期逐日：實際、Naive、四組預測平均與標準差 |
| `predictions_priority_long.csv` | 四組 × 每個 seed × 全部日期 |
| `predictions_test_all_runs.csv` | 所有實驗的測試期逐日預測 |
| `indicator_marginal_effect.csv` | 每個指標加入後的 MAPE 變化 |
| `predictions/`、`loss/` | 每次實驗的逐日預測、每個 epoch 的 loss |
| `figures/` | 圖 1–8 |